In [ ]:
import os

import optuna
import pandas as pd
from common import (
    DATASETS,
    METHODS,
    get_dataset_label,
    get_method_label,
)

In [ ]:
optuna_storage = optuna.storages.RDBStorage(
    os.environ.get("OPTUNA_STORAGE"), skip_table_creation=False
)

In [ ]:
rows = []
for dataset in DATASETS:
    for method in METHODS:
        study_name = f"bayescl/hp/{dataset}/{method}"
        try:
            study = optuna.load_study(study_name=study_name, storage=optuna_storage)
            df = study.trials_dataframe()
        except KeyError:
            print(f"Study {study_name} not found.")
            continue
        # Compute mean, min, and max duration for each trial
        start_times = df["datetime_start"].dt.tz_localize(
            "Pacific/Auckland", ambiguous=True
        )
        end_times = df["datetime_complete"].dt.tz_localize(
            "Pacific/Auckland", ambiguous=False
        )
        durations = end_times - start_times
        rows.append(
            {
                "dataset": dataset,
                "method": method,
                # "min_duration": durations.min().total_seconds() / 3600,
                # "mean_duration": durations.mean().total_seconds() / 3600,
                # "max_duration": durations.max().total_seconds() / 3600,
                "total": durations.sum().total_seconds() / 3600,
            }
        )
        print(
            f"{dataset} - {method}: {durations.min()}, {durations.mean()}, {durations.max()}"
        )

In [ ]:
import tabulate

# Convert to hours and tabulate 2 sf

display_table = []
for row in rows:
    display_table.append(
        {
            "dataset": get_dataset_label(row["dataset"]),
            "method": get_method_label(row["method"]),
            "total": row["total"],
        }
    )
df = pd.DataFrame(display_table)

# pivot table
pivot_df = df.pivot(index="method", columns="dataset", values="total")
pivot_df

In [ ]:
df["total"].sum() / 6

In [ ]:
# to latex
print(tabulate.tabulate(pivot_df, headers="keys", tablefmt="latex", floatfmt=".1f"))

In [ ]:
df = pd.read_parquet("dataframe/summary.parquet")
df = df.groupby(["dataset", "method"]).aggregate({"duration_s": ["mean", "std"]})
# to hours
df["duration_h_mean"] = df["duration_s"]["mean"] / 3600
df["duration_h_std"] = df["duration_s"]["std"] / 3600

# Combine std into mean column
df["duration_h"] = (
    df["duration_h_mean"].round(1).astype(str)
    + "$\\pm$"
    + df["duration_h_std"].round(1).astype(str)
)
df = df.reset_index()
df = df[["dataset", "method", "duration_h"]]
# pivot table
pivot_df = df.pivot(index="method", columns="dataset", values="duration_h")
# pandas to latex
pivot_df.to_latex("duration_table.tex", index=False)

In [ ]:
df["duration_h"]